In [17]:
from langgraph.graph import StateGraph ,START,END
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field
from  dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage,BaseMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
import operator

In [2]:
load_dotenv()
model = ChatGroq(model="openai/gpt-oss-20b")

In [3]:
class ChatState(TypedDict):
    
    messages: Annotated[list[BaseMessage], add_messages]

In [11]:
def chat_node(state: ChatState):

    # take user query from state
    messages = state['messages']

    # send to llm
    response = model.invoke(messages)

    # response store state
    return {'messages': [response]}

In [20]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

#add nodes
graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

chatbot = graph.compile(checkpointer=checkpointer)

In [13]:
initial_state = {
    'messages': [HumanMessage(content="what is the captial of india")]
}
chatbot.invoke(initial_state)['messages'][-1].content

'The capital of India is **New\u202fDelhi**.'

In [21]:
thread_id = '1'
while True:
    user_message = input("Type Here: ")
    
    print('User:',user_message)
    
    if user_message.strip().lower() in ['exit','quit','bye']:
        break
    config = {'configurable':{'thread_id': thread_id}}
    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]},config=config)
    
    print('AI',response['messages'][-1].content)

User: hello my name is vighnesh
AI Hello Vighnesh! 👋 How can I help you today?
User: what is my name is 
AI You mentioned earlier that your name is **Vighnesh**. Is that correct? If you’d like to share anything else or ask a question, feel free!
User: 100*5
AI \(100 \times 5 = 500\)
User: add more 300
AI Sure! \(500 + 300 = 800\)
User: exit


In [22]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='hello my name is vighnesh', additional_kwargs={}, response_metadata={}, id='ee90a5fd-bd51-4db0-aa90-5d2f269d0b85'), AIMessage(content='Hello Vighnesh! 👋 How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "hello my name is vighnesh". We need to respond politely. There\'s no special instruction. We should greet them. Probably ask how we can help.'}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 79, 'total_tokens': 140, 'completion_time': 0.074642115, 'completion_tokens_details': {'reasoning_tokens': 37}, 'prompt_time': 0.003814723, 'prompt_tokens_details': None, 'queue_time': 0.31548485, 'total_time': 0.078456838}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_1074f9ce08', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f8e76-a71a-7a30-a862-459eaff48916-0', tool_calls=[], invalid_tool_call